Phase 6 — Failure Analysis
===========================

Phase 5 produced scores. This phase asks what to DO about them: for every
failing question, which component owns the fix?

  retrieval   evidence never reached the generator (or only part did)
  generation  evidence was there and the answer was still wrong
  corpus      the knowledge base itself is the root cause (tagged, not
              exclusive — a question can fail via retrieval AND touch a
              contradictory article)

Everything here runs without an API key, on the extractive baseline. If
`reports/results/llm_eval.json` exists (written by
`scripts/run_llm_eval.py`), the LLM variants are analysed the same way and
shown alongside — so the section structure does not change when a key is
added, only the number of rows.

Sections
--------
  A. Failure taxonomy for the keyless baseline
  B. Where failures concentrate — by category and by corpus flaw
  C. LLM variants, if results exist
  D. What to fix first

In [ ]:
import json
import sys
from pathlib import Path

# Locate the project root without assuming where Jupyter was launched.
#
# Two wrong assumptions were made here before, each breaking a real
# workflow:
#
#   Path.cwd().parent   assumes the kernel started in notebooks/. Launch
#                       Jupyter from the project root and it resolves one
#                       level too high.
#   walking up only     assumes the kernel started at or below the
#                       project. Launch Jupyter from a folder holding
#                       several projects — a normal thing to do — and the
#                       project is BELOW cwd, so upward search never
#                       finds it.
#
# So: search up, then search down. The marker is a file unique to this
# project, not a generic name like src/ that a sibling project might
# also have.
_MARKER = Path("src") / "corpus" / "seed_articles.py"


def _find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:          # up
        if (candidate / _MARKER).is_file():
            return candidate

    for pattern in ("*", "*/*"):                       # then down, bounded
        found = sorted({m.parents[2]
                        for m in start.glob(f"{pattern}/{_MARKER.as_posix()}")})
        if len(found) == 1:
            return found[0]
        if len(found) > 1:
            names = ", ".join(p.name for p in found)
            raise RuntimeError(
                f"Found several candidate project roots below {start}: "
                f"{names}. Start Jupyter inside the one you want."
            )

    raise RuntimeError(
        f"Could not locate the project root from {start}. Expected to find "
        f"{_MARKER.as_posix()} at, above, or within two levels below it."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
import os
os.chdir(PROJECT_ROOT)          # so data/ and reports/ paths resolve

import matplotlib.pyplot as plt
import pandas as pd

from src.corpus.build import load_corpus, load_golden_set
from src.evaluation.failure_analysis import (
    OWNER,
    classify_run,
    render_markdown,
    summarise,
)
from src.evaluation.rag_metrics import evaluate_run
from src.generation.extractive import ExtractiveAnswerer
from src.generation.pipeline import RAGPipeline
from src.retrieval.chunking import STRATEGIES
from src.retrieval.retrievers import BM25Retriever

FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = Path("reports/results/llm_eval.json")

STRATEGY, DEPTH, THRESHOLD = "markdown_section", 15, 0.35

articles = load_corpus()
golden = load_golden_set()
retriever = BM25Retriever(STRATEGIES[STRATEGY](articles))

## A. Failure taxonomy

In [ ]:
print("=" * 78)
print("A. FAILURE TAXONOMY — extractive baseline, markdown_section + BM25 @ 15")
print("=" * 78)

pipe = RAGPipeline(retriever, ExtractiveAnswerer(min_score=THRESHOLD), depth=DEPTH)
evaluated = evaluate_run(pipe.run(golden), judge=None)
records = classify_run(evaluated)
summary = summarise(records)

print(f"\n  {summary.n} questions, {summary.n_failures} failures "
      f"({summary.n_failures / summary.n:.0%})\n")
print(f"  {'cause':<20}{'owner':<13}{'n':>4}")
print("  " + "-" * 37)
for cause, n in summary.by_cause.most_common():
    print(f"  {cause:<20}{OWNER[cause]:<13}{n:>4}")

## B. Where failures concentrate

In [ ]:
print("\n" + "=" * 78)
print("B. WHERE FAILURES CONCENTRATE")
print("=" * 78)

df = pd.DataFrame([{"category": r.category, "cause": r.cause} for r in records])
ct = pd.crosstab(df["category"], df["cause"])
print("\n" + ct.to_string())

if summary.flaw_exposure:
    print("\n  Questions whose ground truth includes a documented corpus flaw:\n")
    for flaw, c in sorted(summary.flaw_exposure.items()):
        tot = c.get("failed", 0) + c.get("passed", 0)
        print(f"    {flaw:<32} {c.get('failed', 0)}/{tot} failed")

fig, ax = plt.subplots(figsize=(8, 4.5))
ct.plot.barh(stacked=True, ax=ax, colormap="tab20")
ax.set_xlabel("questions")
ax.set_title("Failure causes by question category (extractive baseline)")
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_failure_analysis.png", dpi=130)
plt.close(fig)

## C. LLM variants, if results exist

In [ ]:
print("\n" + "=" * 78)
print("C. LLM VARIANTS")
print("=" * 78)

if RESULTS.exists():
    data = json.loads(RESULTS.read_text(encoding="utf-8"))
    print(f"\n  Generation {data['config']['generation_model']}, "
          f"judge {data['config']['judge_model']}\n")
    print(f"  {'variant':<18}{'refuse_oos':>11}{'answer_in':>11}{'faith':>8}{'rel':>7}")
    for name, v in data["variants"].items():
        g, e = v["generation"], v["evaluation"]
        print(f"  {name:<18}{g['refusal_rate_oos']:>11.1%}"
              f"{g['answer_rate_in_scope']:>11.1%}"
              f"{e['faithfulness']:>8.3f}{e['relevancy']:>7.3f}")
else:
    print("""
  No LLM results found (reports/results/llm_eval.json).

  This is expected without a credential. To produce them:

      python scripts/run_llm_eval.py --dry-run     # cost estimate
      python scripts/run_llm_eval.py               # real run

  Until then, no claim in this repo is made about LLM answer quality.
""")

## D. What to fix first

In [ ]:
print("=" * 78)
print("D. WHAT TO FIX FIRST")
print("=" * 78)
owners = summary.by_owner
print(f"\n  failures by owner: retrieval={owners.get('retrieval', 0)}, "
      f"generation={owners.get('generation', 0)}\n")

out = Path("reports/failure_analysis.md")
out.write_text(render_markdown(summary, "Failure analysis: extractive baseline"),
               encoding="utf-8")
print(f"  wrote {out}")